In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table, ctx
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Configure the CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "Zues9616!"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png' # Grazioso Salvare’s logo 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Defining UI layout
app.layout = html.Div([
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode())),
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard - By Kimani Muhammad'))),
    html.Hr(),
    html.Div(className='filter_options', # Code for the interactive filtering options.
             style={'display' : 'flex'},
             children=[
                 html.Button(id='water-filter', n_clicks=0, children='Water Rescue'),
                 html.Button(id='mountain-filter', n_clicks=0, children='Mountain or Wilderness'),
                 html.Button(id='disaster-filter', n_clicks=0, children='Disaster or Individual Tracking'),
                 html.Button(id='reset-filter', n_clicks=0, children='Reset')
             ]),
    
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         editable=False,
                         row_selectable='single',
                         row_deletable=False,
                         filter_action="native",
                         sort_action="native",
                         sort_mode="multi",
                         selected_columns=[],
                         selected_rows=[0],
                         page_action="native",
                         page_current=0,
                         page_size=10 
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            ),
        html.Div(
            id='graph-id',
            className='col s12 m6',
        )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('water-filter', 'n_clicks'),
              Input('mountain-filter', 'n_clicks'),
              Input('disaster-filter', 'n_clicks'),
              Input('reset-filter', 'n_clicks')])
def update_dashboard(waterFilter, mountainFilter, disasterFilter, resetFilter):
    df = pd.DataFrame.from_records(db.read({}))
    # Uses the triggered id from the dash context to determine which button is clicked
    if (ctx.triggered_id == 'water-filter'):
        df = pd.DataFrame.from_records(db.read({"breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
                                               "sex_upon_outcome": "Intact Female",
                                               "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}))
    elif (ctx.triggered_id == 'mountain-filter'):
        df = pd.DataFrame.from_records(db.read({"breed": {"$in": ["German Shepard", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
                                               "sex_upon_outcome": "Intact Male",
                                               "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}))
    elif (ctx.triggered_id == 'disaster-filter'):
        df = pd.DataFrame.from_records(db.read({"breed": {"$in": ["Doberman Pinscher", "German Shepard", "Golden Retriever", "Bloudhound", "Rottweiler"]},
                                               "sex_upon_outcome": "Intact Male",
                                               "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}}))
    elif (ctx.triggered_id == 'reset-filter'):
        df.drop(columns=['_id'], inplace=True)
        return df.to_dict('records')

    df.drop(columns=['_id'], inplace=True)
    return df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    
    df = viewData
    return [
       dcc.Graph(            
           figure = px.pie(df, names='breed', title='Preferred Animals', color_discrete_sequence=px.colors.sequential.RdBu)
       )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://aliasnadia-edwardborder-3000.codio.io/proxy/8050/
